# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and accessed from the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata name and description
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and fields using their `@id`s as provided by the schema.

In [ ]:
# List all record sets with their `@id` and name
if hasattr(dataset, 'record_sets'):
    record_sets_metadata = dataset.record_sets
else:
    record_sets_metadata = dataset.record_set  # fallback for older versions

record_set_ids = []
print('Available Record Sets and their @id:')
for rs in dataset.list_record_sets():
    print(f"- {rs['@id']}: {rs.get('name', 'No name')}" )
    record_set_ids.append(rs['@id'])
print('\n-- Fields for each Record Set --')
for rs in dataset.list_record_sets():
    rsid = rs['@id']
    rs_meta = dataset.get_record_set(rsid)
    print(f"\nRecord Set: {rsid} (name: {rs.get('name', 'No name')})")
    if hasattr(rs_meta, 'fields'):
        fields = rs_meta.fields
    elif hasattr(rs_meta, 'field'):
        fields = rs_meta.field
    else:
        fields = []
    for f in fields:
        print(f"  - {f['@id']}: {f.get('name', 'No name')} (type: {f.get('dataType', 'unknown')})")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. All entities (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
# Collect all record set @ids (from the previous cell)
record_sets = record_set_ids

dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records for record set {record_set_id}")
        else:
            print(f"No records found for record set {record_set_id}")
    except Exception as ex:
        print(f"Could not load records for {record_set_id}: {ex}")

if dataframes:
    sample_record_set_id = next(iter(dataframes))  # Pick first loaded record set for demonstration
    print(f"\nColumns in {sample_record_set_id}:")
    print(dataframes[sample_record_set_id].columns.tolist())
    print(dataframes[sample_record_set_id].head())
else:
    print("No dataframes loaded. Check record set definition or data availability.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps, such as filtering, normalization, and grouping, using fields by their `@id`.

In [ ]:
# For demonstration, select a numeric field from the loaded record set.
import numpy as np

if dataframes:
    df = dataframes[sample_record_set_id]
    # Identify a numeric field based on dtype
    numeric_field_id = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break
    if numeric_field_id is not None:
        print(f"Using numeric field @{numeric_field_id} for demonstration.")
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with @{numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() or 1)
        print(f"\nNormalized @{numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a non-numeric field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by @{group_field_id} (showing means):")
            print(grouped_df.head())
        else:
            print("No suitable grouping field was found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field or the relationship between two fields, referencing fields by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if dataframes and numeric_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of @{numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"@{numeric_field_id} by @{group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("No visualization available due to missing numeric or grouping field.")

## 6. Conclusion
In this notebook, we demonstrated:
- How to load dataset metadata from a Croissant schema using `mlcroissant`.
- How to enumerate record sets and fields by their `@id`.
- Loading and exploring records in pandas DataFrames referenced by record set `@id`.
- Basic EDA using numeric and categorical fields by `@id`.
- Visualizing data distributions.

Further exploration could include analyzing relationships between specific predictors and adoption outcomes, or incorporating additional field-level documentation from the Croissant schema.